# LeWM: LeWorldModel desde píxeles
`python -m src.cli lewm <acción>` en un notebook: cada sección es una acción de la CLI, en el mismo orden y con los mismos defaults, más una sección propia del modelo. ViT-Tiny + predictor autoregresivo con SIGReg contra el colapso, fiel al repo oficial. Predice el embedding del próximo paso (4 Hz) y planea con CEM. No genera video: su decoder de píxeles se entrena aparte, solo para figuras.

Guía completa: `Docs/modelos/lewm.md`. Acciones de `lewm`: `receta`, `entrenar`, `continuar`, `evaluar`, `bench`, `demo`, `exportar`.

**Cómo se usa.** Con `DRY_RUN = True` (lo de la primera celda) cada acción solo imprime los comandos que correría, igual que `--dry-run`. Con `DRY_RUN = False` corre de verdad por `Toolkit`, en segundo plano (el kernel sigue libre), con candado de GPU y cola, y queda en `results/bitacora.jsonl` con su línea de CLI exacta. Lo que todavía no existe (resultados, checkpoints) se muestra como *pendiente*.

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path

# la raíz del repo: sube desde donde esté abierto el notebook hasta encontrar src/common.py
RAIZ = next((d for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (d / "src" / "common.py").exists()), None)
if RAIZ is None:
  raise SystemExit("abrí el notebook dentro del repo (en Colab, corré antes la primera celda de notebooks/colab.ipynb)")
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

from src import nbui
from src.toolkit import Toolkit

# scripts/run_notebooks.py pone ML2_NB_LIMITE para correr todo chico; a mano queda en None (sin recorte)
LIMITE = int(os.environ.get("ML2_NB_LIMITE", "0")) or None

In [ ]:
MODELO = "lewm"
RUN = "lewm"       # nombre de la corrida: <modelo>_<tamaño>[_<variante>][_s<semilla>]
DRY_RUN = True      # True: solo imprime los comandos (= --dry-run). False: los corre de verdad, en segundo plano
DEVICE = nbui.device_inicial()      # el que detecta el repo: cuda, rocm, mps, cpu (dml y npu: con ONNX Runtime)
DEV = nbui.device_kw(DEVICE)        # = --device
tk = Toolkit(MODELO)
print(MODELO, "| acciones:", ", ".join(tk.acciones), "| corrida:", RUN, "| device:", DEVICE, "| dry-run:", DRY_RUN)

## 1. Receta y fidelidad
La receta de entrenamiento es la de la fuente del modelo (con lo forzado por datos o presupuesto marcado) y la tabla de fidelidad dice qué es fiel, qué lo fuerzan los datos o el presupuesto y qué es propio. CLI: `python -m src.cli lewm receta`.

In [ ]:
r = nbui.receta(MODELO, None)        # la Recipe (optimizador, lr por tramos, wd, EMA, clip, batch, presupuesto) y lo forzado
nbui.fidelidad(MODELO);                # tabla de fidelidad: decisión, fuente, lo que hace el repo, categoría

In [ ]:
nbui.opciones(MODELO, "receta", )   # opciones de la acción y el comando equivalente
t = nbui.lanzar(MODELO, "receta", dry_run=DRY_RUN, )   # lo que imprime `python -m src.recipes`

## 2. Entrenar o continuar
Entrenar de cero con la receta de la fuente, probar el lr en un humo corto, seguir la corrida en vivo y continuarla o extenderla. CLI: `entrenar` (con `--humo`), `continuar`, `estado`.

In [ ]:
OPC_ENTRENAR = dict(limite=LIMITE)     # los mismos nombres que la CLI: --tamaño, --destilar, --variante, --limite...
nbui.opciones(MODELO, "entrenar", **OPC_ENTRENAR);

In [ ]:
# humo: 300 pasos con el lr de la fuente, escalado por raíz, lineal, x3 y x10; elige el que mejor baja (--humo)
t = nbui.lanzar(MODELO, "entrenar", dry_run=DRY_RUN, humo=True, **dict())

In [ ]:
# entrenamiento de verdad (con DRY_RUN = False corre en segundo plano: el kernel sigue libre)
t = nbui.lanzar(MODELO, "entrenar", dry_run=DRY_RUN, **OPC_ENTRENAR)

In [ ]:
# seguimiento: barra con porcentaje y ETA, y curvas de loss, lr y monitoreo con el tramo del schedule sombreado
nbui.mostrar_barra(RUN)
nbui.curvas(RUN)
# nbui.en_vivo(RUN)     # se actualiza solo hasta que la corrida termina (en una corrida real)

In [ ]:
nbui.estado_corridas()                                 # una fila por corrida de checkpoints/ (= python -m src.status)
nbui.panel_cola();                                      # si la GPU estaba ocupada, la acción quedó en la cola

**Continuar.** Retomar una corrida cortada (`continuar --run`) o extenderla con más presupuesto (`--presupuesto 2x`): la extensión es una corrida nueva `<run>_x<k>` con `parent`, sha y receta. Las recetas con coseno solo se extienden con `--restart`.

In [ ]:
OPC_CONTINUAR = dict(run=RUN, limite=LIMITE)                       # retomar desde last.pt
OPC_EXTENDER = dict(run=RUN, presupuesto="2x", limite=LIMITE)      # extender: corrida nueva <run>_x2
nbui.opciones(MODELO, "continuar", **OPC_CONTINUAR)
t = nbui.lanzar(MODELO, "continuar", dry_run=DRY_RUN, **OPC_CONTINUAR)
t = nbui.lanzar(MODELO, "continuar", dry_run=DRY_RUN, **OPC_EXTENDER)

## 3. Evaluar
Evaluación latente: error de predicción por horizonte contra copiar el último latente, probes contra el encoder sin entrenar y contra píxeles, sorpresa y plan con CEM. CLI: `python -m src.cli lewm evaluar`.

In [ ]:
OPC_EVALUAR = dict(run=RUN, limite=LIMITE, **DEV)
nbui.opciones(MODELO, "evaluar", **OPC_EVALUAR)
t = nbui.lanzar(MODELO, "evaluar", dry_run=DRY_RUN, **OPC_EVALUAR)

In [ ]:
nbui.tabla_latente(solo=[RUN]);        # una fila con las referencias al lado; "pendiente" hasta que exista latent_<run>.json

## 4. Latencia
Un protocolo único: un paso por frame en batch 1, fp16 con CUDA graphs donde anda, p50 y p99; con `--cadena` mide la cadena cámara → encoder → modelo → decoder, en bucle cerrado y a ritmo de cámara (50 ms). CLI: `python -m src.cli lewm bench`.

In [ ]:
OPC_BENCH = dict(run=RUN, limite=LIMITE, **DEV)
nbui.opciones(MODELO, "bench", **OPC_BENCH)
t = nbui.lanzar(MODELO, "bench", dry_run=DRY_RUN, **OPC_BENCH)                                   # el modelo solo
t = nbui.lanzar(MODELO, "bench", dry_run=DRY_RUN, cadena=True, ritmo=50, **OPC_BENCH)            # la cadena a ritmo de cámara
t = nbui.lanzar(MODELO, "bench", dry_run=DRY_RUN, cadena=True, diagnostico=True, **OPC_BENCH)    # el piso de ~20 ms a ritmo de cámara

La latencia no depende de los pesos: con `aleatorio=True` se mide la arquitectura sin entrenar.

In [ ]:
t = nbui.lanzar(MODELO, "bench", dry_run=DRY_RUN, aleatorio=True, limite=LIMITE, **DEV)

In [ ]:
nbui.latencia_etapas();      # p50 y p99 de cada columna del protocolo y mediana por etapa, de results/infer/

## 5. Demo
Decodificación del latente con el decoder de píxeles, máscaras (V-JEPA 2) y curva de error. CLI: `python -m src.cli lewm demo`.

In [ ]:
OPC_DEMO = dict(run=RUN, limite=LIMITE, **DEV)
nbui.opciones(MODELO, "demo", **OPC_DEMO)
t = nbui.lanzar(MODELO, "demo", dry_run=DRY_RUN, **OPC_DEMO)          # figura y curvas de error (src.infer.demo_latente)
nbui.figura(nbui.ultimo("*_latente.png"));

In [ ]:
# el decoder de píxeles se entrena aparte (encoder congelado, MSE): solo para figuras
t = nbui.lanzar(MODELO, "demo", dry_run=DRY_RUN, entrenar_decoder=True, limite=LIMITE, **DEV, **({"encoder": "vjepa2_s"} if MODELO == "vjepa2_ac" else {}))

## 6. Exportar
ONNX, donde aplica (`python -m src.cli lewm exportar`). El export verifica que ONNX Runtime coincida con PyTorch.

In [ ]:
OPC_EXPORTAR = dict(run=RUN) if MODELO != "codec" else dict()
nbui.opciones(MODELO, "exportar", **OPC_EXPORTAR)
t = nbui.lanzar(MODELO, "exportar", dry_run=DRY_RUN, **OPC_EXPORTAR)
t = nbui.lanzar(MODELO, "exportar", dry_run=DRY_RUN, dtype="fp16", **OPC_EXPORTAR)      # la versión fp16

## 7. Sección propia: SIGReg, sorpresa y CEM
**SIGReg** (λ 0,09, 17 nodos, 1024 proyecciones) empuja los embeddings hacia una gaussiana isótropa y evita el colapso: en las curvas se ve junto con el error de predicción y el desvío de los latentes (`z_std`). La **sorpresa** es el error de predicción cuando el video se corta (teletransporte o salto de 2 s). El **CEM** planea las acciones que llevan al latente de un frame objetivo.

In [ ]:
nbui.curvas(RUN);     # loss, pred_mse, sigreg y z_std (si el log los trae)

In [ ]:
nbui.sorpresa_fig(RUN);

In [ ]:
nbui.horizonte_latente(RUN);

In [ ]:
nbui.cem_tabla(RUN);

In [ ]:
t = nbui.lanzar(MODELO, "evaluar", dry_run=DRY_RUN, run=RUN, limite=LIMITE, **DEV)           # la evaluación completa
t = nbui.lanzar(MODELO, "evaluar", dry_run=DRY_RUN, run=RUN, sin_plan=True, limite=LIMITE, **DEV)   # sin el CEM (más rápido)